# Size and value in China 复现

先构造 Liu、Stambaugh 和 Yuan（2019）的 CH-3，再用完全相同的 2×3 组合逻辑构造 FF-3，最后复现 Table 3 和 Table 5。

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

current_path = Path.cwd()
project_root = next(
    path for path in [current_path, *current_path.parents]
    if (path / "README.md").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

with (project_root / "config.local.json").open(encoding="utf-8") as file:
    data_root = Path(json.load(file)["data_root"])

print("数据目录存在：", data_root.exists())

数据目录存在： True


## 1. 复现口径

样本期是 2000 年 1 月至 2016 年 12 月。每个月末先应用上市时间和交易记录筛选，再删除按上月末规模排序最小的 30%，对剩余股票独立进行 2×3 排序。

- 由于数据源缺失A股股本(S_SHARE_TOTALA)，此处使用当日总股本(TOT_SHR_TODAY)进行替换。`size_me = 月末收盘价 × TOT_SHR_TODAY × 10,000`，并用它做微盘剔除、规模排序和组合权重。本次复现中，主要是VMG的差异较大，数据缺失可能是出现差异的重要原因之一。
- `EP_raw = 最新可得扣非净利润 / (月末收盘价 × 总股本)`；扣非净利润缺失时使用普通净利润。用于排序的 `EP = max(EP_raw, 0)`。
- 财报首先使用实际公告日期，缺失时使用公告日期；月末当天公告视为当月末可得。2002 年以前仅使用半年报和年报，2002 年起使用季报。
- 原文使用最近一个月作为口径，但是在实际测试中发现，因为春节期间存在大量无交易日期，一月二月经常不存在有效股票数，故使用最近 22 个市场交易日中，个股至少有 15 条交易记录进行替代；这一窗口不按自然月切分。
- 一年期存款利率按复利折算为月利率。
FF-3 与 CH-3 的区别只应是估值变量：CH-3 使用经过非负处理的 EP，FF-3 使用 `BM = book equity / market value of total shares`。其他设定保持一致。

In [2]:
sample_start = "2000-01-01"
sample_end = "2016-12-31"
history_start = "1999-01-01"
formation_start = "1999-12-01"
formation_end = "2016-11-30"
financial_history_start = "1998-01-01"
stock_prefixes = ("60", "00", "30")

minimum_listing_months = 6
minimum_active_days_lookback = 120
recent_trading_day_window = 22
minimum_active_days_recent_window = 15
activity_lookback_months = 12
active_trade_values = ("交易",)
preferred_statement_types = ("408005000", "408001000")
quarterly_reporting_start = "2002-01-01"
pre_quarterly_report_months = (6, 12)

microcap_quantile = 0.30
size_quantile = 0.50
value_quantiles = (0.30, 0.70)
total_share_multiplier = 10_000.0
annual_rate_divisor = 100.0
periods_per_year = 12
csv_chunksize = 500_000
csv_encoding = "utf-8-sig"

## 2. 数据处理与加载

In [3]:
data_files = {
    "日行情": data_root / "A股日频数据" / "ASHAREEODPRICES_202608041042.csv",
    "市值与股本": data_root / "A股日频数据" / "ASHAREEODDERIVATIVEINDICATOR_202608031334.csv",
    "利润表": data_root / "财务指标数据" / "ASHAREINCOME_202607311449.csv",
    "资产负债表": data_root / "财务指标数据" / "ASHAREBALANCESHEET_202607311406.csv",
}
required_columns = {
    "日行情": ["S_INFO_WINDCODE", "TRADE_DT", "S_DQ_ADJPRECLOSE", "S_DQ_ADJCLOSE", "S_DQ_TRADESTATUS"],
    "市值与股本": ["S_INFO_WINDCODE", "TRADE_DT", "S_DQ_CLOSE_TODAY", "TOT_SHR_TODAY"],
    "利润表": ["S_INFO_WINDCODE", "ACTUAL_ANN_DT", "ANN_DT", "REPORT_PERIOD", "STATEMENT_TYPE", "NET_PROFIT_AFTER_DED_NR_LP", "NET_PROFIT_EXCL_MIN_INT_INC"],
    "资产负债表": ["S_INFO_WINDCODE", "ACTUAL_ANN_DT", "ANN_DT", "REPORT_PERIOD", "STATEMENT_TYPE", "TOT_SHRHLDR_EQY_EXCL_MIN_INT"],
}

file_check = []
for name, path in data_files.items():
    columns = pd.read_csv(path, nrows=0, encoding=csv_encoding).columns
    file_check.append({
        "数据表": name,
        "文件大小（GB）": round(path.stat().st_size / 1024**3, 2),
        "缺少字段": [column for column in required_columns[name] if column not in columns],
    })
display(pd.DataFrame(file_check))

,数据表,文件大小（GB）,缺少字段
0,日行情,4.29,[]
1,市值与股本,10.78,[]
2,利润表,1.14,[]
3,资产负债表,0.65,[]


## 3. 构造 PIT 月度面板

形成月的市值、EP、上市时间和过去交易记录只与下一月收益连接。财报使用向后匹配，要求 `available_date <= formation_month`，避免使用形成月末之后公布的信息。按照论文 Appendix A.1，2002 年以前只允许半年报和年报进入匹配，2002 年起才使用季报。

In [4]:
from data import (
    add_point_in_time_book_equity,
    build_monthly_panel,
    load_month_end_characteristics,
    load_monthly_returns,
    load_monthly_risk_free,
    load_or_build,
    load_point_in_time_book_equity,
    load_point_in_time_earnings,
    load_recent_trading_records,
)

cache_dir = project_root / "work" / "ch3_total_share_cache"
monthly_returns = load_or_build(
    cache_dir / "monthly_returns.parquet",
    lambda: load_monthly_returns(
        data_files["日行情"], history_start=history_start, sample_end=sample_end,
        stock_prefixes=stock_prefixes, active_trade_values=active_trade_values,
        lookback_months=activity_lookback_months, chunksize=csv_chunksize, encoding=csv_encoding,
    ),
)
recent_trading_records = load_or_build(
    cache_dir / "recent_22d_trading_records.parquet",
    lambda: load_recent_trading_records(
        data_files["日行情"], formation_start=formation_start, formation_end=formation_end,
        stock_prefixes=stock_prefixes, active_trade_values=active_trade_values,
        trading_day_window=recent_trading_day_window, chunksize=csv_chunksize, encoding=csv_encoding,
    ),
)
month_end_characteristics = load_or_build(
    cache_dir / "month_end_characteristics.parquet",
    lambda: load_month_end_characteristics(
        data_files["市值与股本"], formation_start=formation_start, formation_end=formation_end,
        stock_prefixes=stock_prefixes, total_share_multiplier=total_share_multiplier,
        chunksize=csv_chunksize, encoding=csv_encoding,
    ),
)
earnings = load_or_build(
    cache_dir / "point_in_time_earnings.parquet",
    lambda: load_point_in_time_earnings(
        data_files["利润表"], report_start=financial_history_start, available_end=formation_end,
        stock_prefixes=stock_prefixes, statement_types=preferred_statement_types,
        chunksize=csv_chunksize, encoding=csv_encoding,
    ),
)
book_equity = load_or_build(
    cache_dir / "point_in_time_book_equity.parquet",
    lambda: load_point_in_time_book_equity(
        data_files["资产负债表"], report_start=financial_history_start, available_end=formation_end,
        stock_prefixes=stock_prefixes, statement_types=preferred_statement_types,
        chunksize=csv_chunksize, encoding=csv_encoding,
    ),
)
risk_free = load_monthly_risk_free(
    data_root / "银行利率文件151306120" / "CSR_Intrst.xlsx",
    annual_rate_divisor=annual_rate_divisor, periods_per_year=periods_per_year,
)
monthly_panel = load_or_build(
    cache_dir / "monthly_panel_recent22_semiannual_pre2002.parquet",
    lambda: build_monthly_panel(
        monthly_returns, month_end_characteristics, earnings, recent_trading_records,
        sample_start=sample_start, sample_end=sample_end,
        minimum_listing_months=minimum_listing_months,
        minimum_active_days_lookback=minimum_active_days_lookback,
        minimum_active_days_recent_window=minimum_active_days_recent_window,
        quarterly_reporting_start=quarterly_reporting_start,
        pre_quarterly_report_months=pre_quarterly_report_months,
    ),
)
monthly_panel = add_point_in_time_book_equity(
    monthly_panel, book_equity,
    quarterly_reporting_start=quarterly_reporting_start,
    pre_quarterly_report_months=pre_quarterly_report_months,
)

/opt/anaconda3/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [5]:
monthly_panel["EP_raw"] = monthly_panel["earnings"] / monthly_panel["valuation_me"]
monthly_panel["EP"] = monthly_panel["EP_raw"].clip(lower=0)
monthly_panel["BM"] = monthly_panel["book_equity"] / monthly_panel["valuation_me"]
monthly_panel["BM_for_FF3"] = monthly_panel["BM"].where(
    monthly_panel["EP"].notna() & monthly_panel["BM"].gt(0)
)
monthly_panel["base_eligible"] = (
    monthly_panel["passes_listing_age_filter"]
    & monthly_panel["passes_trading_filter"]
    & monthly_panel["size_me"].gt(0)
    & monthly_panel["stock_return"].notna()
)

assert monthly_panel.loc[monthly_panel["EP_raw"].lt(0), "EP"].eq(0).all()
assert monthly_panel.loc[monthly_panel["BM_for_FF3"].notna(), "BM"].gt(0).all()
assert monthly_panel.loc[monthly_panel["BM_for_FF3"].notna(), "EP"].notna().all()
assert not (monthly_panel["available_date"] > monthly_panel["formation_month"]).fillna(False).any()
assert not (monthly_panel["book_available_date"] > monthly_panel["formation_month"]).fillna(False).any()
assert not (
    monthly_panel["report_period"].lt(quarterly_reporting_start)
    & ~monthly_panel["report_period"].dt.month.isin(pre_quarterly_report_months)
).any()
assert not (
    monthly_panel["book_report_period"].lt(quarterly_reporting_start)
    & ~monthly_panel["book_report_period"].dt.month.isin(pre_quarterly_report_months)
).any()
assert not monthly_panel.duplicated(["stock_id", "return_month"]).any()

### 3.1 每月股票池筛选

下面展示每一步筛选的样本损失和最近 22 个市场交易日筛选。EP 缺失不影响市场因子，但不能进入六个 Size×EP 组合。

In [6]:
listed = monthly_panel["passes_listing_age_filter"]
trading = monthly_panel["passes_trading_filter"]
positive_size = monthly_panel["size_me"].gt(0)
next_return = monthly_panel["stock_return"].notna()

filter_summary = pd.DataFrame([
    {"步骤": "月度面板", "股票-月数量": len(monthly_panel)},
    {"步骤": "上市至少六个月", "股票-月数量": int(listed.sum())},
    {"步骤": "再满足交易记录要求", "股票-月数量": int((listed & trading).sum())},
    {"步骤": "再要求正市值和下一月收益", "股票-月数量": int((listed & trading & positive_size & next_return).sum())},
    {"步骤": "其中 EP 可用", "股票-月数量": int((monthly_panel["base_eligible"] & monthly_panel["EP"].notna()).sum())},
    {"步骤": "其中 BM 可用", "股票-月数量": int((monthly_panel["base_eligible"] & monthly_panel["BM"].notna()).sum())},
    {"步骤": "其中可进入 FF-3（EP 可用且 BM>0）", "股票-月数量": int((monthly_panel["base_eligible"] & monthly_panel["BM_for_FF3"].notna()).sum())},
])
display(filter_summary)
display(monthly_panel["active_days_recent_22"].describe())

,步骤,股票-月数量
0,月度面板,1003884
1,上市至少六个月,359216
2,再满足交易记录要求,324863
3,再要求正市值和下一月收益,324855
4,其中 EP 可用,324596
5,其中 BM 可用,324855
6,其中可进入 FF-3（EP 可用且 BM>0）,320546


count    1.003884e+06
mean     7.391379e+00
std      1.022059e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      2.200000e+01
max      2.200000e+01
Name: active_days_recent_22, dtype: float64

## 4. 共同的 2×3 组合构造

每月先按 `size_me` 删除最小 30%，再在保留股票中按规模中位数分 S/B，并按非负处理后的 EP 的 30%/70% 分位数分 G/M/V。六个交叉组合内部按 `size_me` 加权：

- `SMB = (SG + SM + SV)/3 - (BG + BM + BV)/3`
- `VMG = (SV + BV)/2 - (SG + BG)/2`
- `MKT = 顶部 70% 股票的市值加权收益 - RF`

同一个 `construct_size_value_factors` 函数以后以 BM、L/M/H 标签和 HML 名称运行，即得到 FF-3。

In [7]:
from factors import construct_size_value_factors

ch3_factors, ch3_six_portfolios, ch3_diagnostics = construct_size_value_factors(
    monthly_panel,
    risk_free,
    characteristic="EP",
    size_factor_name="SMB",
    value_factor_name="VMG",
    value_labels=("G", "M", "V"),
    microcap_quantile=microcap_quantile,
    size_quantile=size_quantile,
    value_quantiles=value_quantiles,
)

display(ch3_diagnostics.head())
display(ch3_six_portfolios.head())
display(ch3_factors[["month", "MKT", "SMB", "VMG", "RF"]].head())

EP portfolios:   0%|          | 0/204 [00:00<?, ?month/s]

,month,eligible_before_microcap_filter,eligible_after_microcap_filter,eligible_for_six_portfolios,microcap_cutoff,size_break,ep_30,ep_70,n_SG,n_SM,n_SV,n_BG,n_BM,n_BV
0,2000-01-31,868,607,605,1.519870e+09,2.691000e+09,0.006963,0.016047,104,127,72,78,114,110
1,2000-02-29,880,616,614,1.691297e+09,3.126958e+09,0.005846,0.014724,99,129,79,85,117,105
2,2000-03-31,885,619,617,1.900424e+09,3.463058e+09,0.005287,0.014156,95,126,88,90,121,97
3,2000-04-30,893,625,623,2.098833e+09,3.749760e+09,0.006788,0.019663,98,128,86,89,121,101
4,2000-05-31,901,630,628,2.141801e+09,3.783049e+09,0.010898,0.023385,103,133,78,86,117,111


,month,BG,BM,BV,SG,SM,SV
0,2000-01-31,0.120180,0.189070,0.124702,0.152112,0.133331,0.135729
1,2000-02-29,0.175731,0.100269,0.084737,0.156650,0.128585,0.100756
2,2000-03-31,0.037189,0.006185,0.049954,0.110311,0.092893,0.083178
3,2000-04-30,0.024028,0.004617,0.036267,-0.006126,0.007687,0.031709
4,2000-05-31,-0.011384,0.020354,0.042230,0.050337,0.030383,0.032935


,month,MKT,SMB,VMG,RF
0,2000-01-31,0.143690,-0.004260,-0.005931,0.001856
1,2000-02-29,0.118297,0.008418,-0.073444,0.001856
2,2000-03-31,0.045633,0.064351,-0.007184,0.001856
3,2000-04-30,0.016027,-0.010548,0.025037,0.001856
4,2000-05-31,0.021643,0.020819,0.018105,0.001856


## 5. Table 3：CH-3 描述性统计

Table 3 的 t 值是月度均值除以其普通标准误；相关系数单独报告。收益均值和标准差的单位均为每月百分比。

In [8]:
from inference import summarize_factors

factor_names = ("MKT", "SMB", "VMG")
replicated_table3 = summarize_factors(ch3_factors, factor_names)
paper_table3 = pd.DataFrame({
    "factor": factor_names,
    "paper_mean_percent": (0.66, 1.03, 1.14),
    "paper_std_percent": (8.09, 4.52, 3.75),
    "paper_t": (1.16, 3.25, 4.34),
})
table3_comparison = replicated_table3.merge(paper_table3, on="factor")
display(table3_comparison.round(3))
factor_corr = ch3_factors[list(factor_names)].corr()
display(factor_corr.round(3))

,factor,months,mean_percent,std_percent,t_statistic,paper_mean_percent,paper_std_percent,paper_t
0,MKT,204,0.693,8.072,1.226,0.66,8.09,1.16
1,SMB,204,0.953,4.492,3.030,1.03,4.52,3.25
2,VMG,204,0.963,4.094,3.362,1.14,3.75,4.34


,MKT,SMB,VMG
MKT,1.000,0.120,-0.261
SMB,0.120,1.000,-0.621
VMG,-0.261,-0.621,1.000


In [9]:
results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)
ch3_factors.to_csv(results_dir / "ch3_factors_total_share.csv", index=False)
ch3_six_portfolios.to_csv(results_dir / "ch3_six_portfolios_total_share.csv", index=False)
ch3_diagnostics.to_csv(results_dir / "ch3_diagnostics_total_share.csv", index=False)
replicated_table3.to_csv(results_dir / "ch3_table3_summary_total_share.csv", index=False)
factor_corr.to_csv(
    results_dir / "ch3_factor_correlations_total_share.csv"
)

## 6. FF-3：用 BM 替换 EP

`book_equity` 取资产负债表的 `TOT_SHRHLDR_EQY_EXCL_MIN_INT`，BM 使用与 EP 相同的总股本市值分母。账面权益也按财报可得日期向后匹配。

FF-3 沿用上一节的共同排序函数，并严格使用 CH-3 的可排序股票池：EP 必须可得，同时按照 Fama–French 的 BM 组合口径要求账面权益为正。原始 `BM` 保留；`BM_for_FF3` 只是在不满足这两个条件时设为缺失，并不把负 BM 截断成零。

六个组合为 `SL/SM/SH/BL/BM/BH`，内部仍按上月末总股本市值加权。`FFSMB` 是三个小盘组合平均收益减去三个大盘组合平均收益，`FFHML` 是两个高 BM 组合平均收益减去两个低 BM 组合平均收益。

In [10]:
ff3_factors, ff3_six_portfolios, ff3_diagnostics = construct_size_value_factors(
    monthly_panel,
    risk_free,
    characteristic="BM_for_FF3",
    size_factor_name="FFSMB",
    value_factor_name="FFHML",
    value_labels=("L", "M", "H"),
    microcap_quantile=microcap_quantile,
    size_quantile=size_quantile,
    value_quantiles=value_quantiles,
)

pd.testing.assert_series_equal(
    ch3_factors.set_index("month")["MKT"],
    ff3_factors.set_index("month")["MKT"],
    check_names=False,
)
display(summarize_factors(ff3_factors, ("MKT", "FFSMB", "FFHML")).round(3))
display(ff3_diagnostics.head())

BM_for_FF3 portfolios:   0%|          | 0/204 [00:00<?, ?month/s]

,factor,months,mean_percent,std_percent,t_statistic
0,MKT,204,0.693,8.072,1.226
1,FFSMB,204,0.589,5.075,1.658
2,FFHML,204,0.874,4.590,2.718


,month,eligible_before_microcap_filter,eligible_after_microcap_filter,eligible_for_six_portfolios,microcap_cutoff,size_break,bm_for_ff3_30,bm_for_ff3_70,n_SL,n_SM,n_SH,n_BL,n_BM,n_BH
0,2000-01-31,868,607,604,1.519870e+09,2.693852e+09,0.177191,0.317363,73,134,95,108,108,86
1,2000-02-29,880,616,613,1.691297e+09,3.129000e+09,0.152892,0.275483,67,138,102,117,107,82
2,2000-03-31,885,619,616,1.900424e+09,3.463369e+09,0.134612,0.249569,63,144,101,122,102,84
3,2000-04-30,893,625,622,2.098833e+09,3.752037e+09,0.127256,0.232350,74,144,93,113,104,94
4,2000-05-31,901,630,626,2.141801e+09,3.787647e+09,0.127603,0.237995,77,136,100,111,114,88


## 7. Table 5：CH-3 与 FF-3 相互解释

Panel A 的四个时间序列回归都包含市场因子和另一模型的规模、价值因子。表中的 alpha 按月度百分比报告，括号对应 White HC0 异方差稳健 t 值。Panel B 用 GRS 检验另一模型的两个因子 alpha 是否联合为零。

In [11]:
from inference import factor_regression, grs_test

table5_data = ch3_factors[["month", "MKT", "SMB", "VMG"]].merge(
    ff3_factors[["month", "FFSMB", "FFHML"]],
    on="month", how="inner", validate="one_to_one",
)
specifications = (
    ("CH-3", "FFSMB", ("MKT", "SMB", "VMG"), -0.04, -0.66),
    ("CH-3", "FFHML", ("MKT", "SMB", "VMG"), 0.34, 0.97),
    ("FF-3", "SMB", ("MKT", "FFSMB", "FFHML"), 0.47, 7.03),
    ("FF-3", "VMG", ("MKT", "FFSMB", "FFHML"), 1.39, 7.93),
)
panel_a_rows = []
for model, dependent, regressors, paper_alpha, paper_t in specifications:
    result = factor_regression(table5_data, dependent, regressors)
    result["benchmark_model"] = model
    result["paper_alpha_percent"] = paper_alpha
    result["paper_alpha_t"] = paper_t
    panel_a_rows.append(result)
table5_panel_a = pd.DataFrame(panel_a_rows)
display(table5_panel_a[[
    "dependent", "benchmark_model", "months", "alpha_percent", "alpha_t",
    "paper_alpha_percent", "paper_alpha_t",
]].round(3))

,dependent,benchmark_model,months,alpha_percent,alpha_t,paper_alpha_percent,paper_alpha_t
0,FFSMB,CH-3,204,-0.053,-0.830,-0.04,-0.66
1,FFHML,CH-3,204,0.539,1.586,0.34,0.97
2,SMB,FF-3,204,0.465,6.875,0.47,7.03
3,VMG,FF-3,204,1.208,6.694,1.39,7.93


In [12]:
panel_b_rows = []
for model, assets, factors, paper_f, paper_p in (
    ("CH-3", ("FFSMB", "FFHML"), ("MKT", "SMB", "VMG"), 0.88, 0.41),
    ("FF-3", ("SMB", "VMG"), ("MKT", "FFSMB", "FFHML"), 33.90, 2.14e-13),
):
    result = grs_test(table5_data[list(assets)], table5_data[list(factors)])
    result["benchmark_model"] = model
    result["tested_factors"] = ", ".join(assets)
    result["paper_grs_f"] = paper_f
    result["paper_p_value"] = paper_p
    panel_b_rows.append(result)
table5_panel_b = pd.DataFrame(panel_b_rows)
display(table5_panel_b[[
    "tested_factors", "benchmark_model", "months", "grs_f", "p_value",
    "paper_grs_f", "paper_p_value",
]])

,tested_factors,benchmark_model,months,grs_f,p_value,paper_grs_f,paper_p_value
0,"FFSMB, FFHML",CH-3,204.0,2.064585,1.295800e-01,0.88,4.100000e-01
1,"SMB, VMG",FF-3,204.0,24.037537,4.460690e-10,33.90,2.140000e-13


In [13]:
ff3_factors.to_csv(results_dir / "ff3_factors_total_share.csv", index=False)
ff3_six_portfolios.to_csv(results_dir / "ff3_six_portfolios_total_share.csv", index=False)
ff3_diagnostics.to_csv(results_dir / "ff3_diagnostics_total_share.csv", index=False)
table5_panel_a.to_csv(results_dir / "table5_panel_a_total_share.csv", index=False)
table5_panel_b.to_csv(results_dir / "table5_panel_b_total_share.csv", index=False)